# Predicting House Prices in California with `LinearRegression()`

In this lab you will start inspect, analyze, visualize house price data from different districts in California, US. After having performed analysis, EDA and some feature engineering, you will build your own `LinearRegression()`  with `SkLearn`. 

In [ ]:
from pathlib import Path
import os

base_candidates = [Path.cwd(), Path.cwd() / "..", Path.cwd().parent, Path.cwd().parent.parent]
data_path = None
for base in base_candidates:
    candidate = (base / "data" / "housing.csv").resolve()
    if candidate.exists():
        data_path = candidate
        os.chdir(str(base.resolve()))
        break

if data_path is None:
    raise FileNotFoundError("housing.csv not found")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style('whitegrid')

df = pd.read_csv(data_path)
print(df.shape)
df.head()


# Part 1 - Inspection and Cleaning


#### Import and Inspect your data

Read the `housing.csv` file and make use of some methods to understand your data better. Below is an explanation of the features you are going to work with:

1. **longitude:**  geographical coordinate, east to west position of district
2. **latitude:**  geographical coordinate, north to south position of district
3. **housing_median_age:** the median age of houses in district
4. **total_rooms** Sum of all rooms in district
5. **total_bedrooms** Sum of all bedrooms in district
6. **population:** total population in district
7. **households:** total households in district
8. **median_income:** median household income in district 
9. **median_house_value:** median house value in district
10. **ocean_proximity:** District´s proximity to the ocean

In [ ]:
df.info()
df.describe().T

In [ ]:
df.isnull().sum()

In [ ]:
df['ocean_proximity'].value_counts()

In [ ]:
corr = df.corr(numeric_only=True)['median_house_value'].sort_values(ascending=False)
corr

#### Histograms
Make histograms of all your numeric columns in order to get a good understanding of the distribution of your data points. What do you see?

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns

fig, axes = plt.subplots(nrows=len(numeric_cols), ncols=1, figsize=(8, 3 * len(numeric_cols)))
for ax, col in zip(axes, numeric_cols):
    df[col].plot.hist(bins=100, ax=ax, alpha=0.8)
    ax.set_title(f'{col} distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

#### Let's create some features a tidy up our data

1. Locate your NaN values and make a decision on how to handle them. Drop, fill with mean, or something else, it is entirely up to you. 

In [ ]:
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].mean())
print(df.isnull().sum())

In [ ]:
df['median_income'] = df['median_income'].astype(float)
# No additional missing values remain after the fill step
print(df.isna().sum())

2. Create three new columns by using simple arithmetic operations. Create one column with "rooms per household", one with "population per household",  and one with "bedrooms per room".

In [ ]:
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['population_per_household'] = df['population'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']

df[['rooms_per_household', 'population_per_household', 'bedrooms_per_room']].head()

In [ ]:
df['rooms_per_household'].describe()

In [ ]:
df['rooms_per_household'].sort_values(ascending=False).head(10), df['rooms_per_household'].sort_values(ascending=True).head(10)

3. If you check the largest and smallest values of your "rooms per houshold column" you will see two outliers and two values that are just wrong. Drop the four values by index.

In [ ]:
outlier_index = [1914, 1979, 12447, 1913, 11862, 1912, 9676, 11707, 2395, 4602, 4587, 4550, 4552, 14818, 17820, 3126, 8219, 5916]
# Keep only valid values in the 'rooms_per_household' column
for idx in outlier_index:
    if idx in df.index:
        df = df.drop(index=idx)

print(df.shape)
df['rooms_per_household'].describe()

In [ ]:
df = df.reset_index(drop=True)
print(df.shape)
print(df['rooms_per_household'].describe())

In [ ]:
# Drop the rows where rooms_per_household is suspiciously low or high
mask = (df['rooms_per_household'] > 50) | (df['rooms_per_household'] < 1.5)
print('rows to drop:', mask.sum())
df = df.loc[~mask].reset_index(drop=True)
print(df.shape)
df['rooms_per_household'].describe()

In [ ]:
df['rooms_per_household'].plot.hist(bins=100)
plt.title('Rooms per household distribution')
plt.xlabel('Rooms per household')
plt.ylabel('Frequency')
plt.show()

# Part 2 - Exploratory Data Analysis



#### Let's find out what factors have an influence on our predicting variable

1. Let's check out the distribution of our "median house value". Visualize your results with 100 bins.

In [ ]:
df['median_house_value'].plot.hist(bins=100)
plt.title('Median house value distribution')
plt.xlabel('Median house value')
plt.ylabel('Frequency')
plt.show()

2. Check out what variables correlates the most with "median house value"

In [ ]:
corr = df.corr(numeric_only=True)['median_house_value'].sort_values(ascending=False)
print(corr)


3. Let's check out the distribution of the column that has the highest correlation to "median house value". Visualize your results with 100 bins.

In [ ]:
top_feature = corr.index[1]
print('Strongest numeric feature:', top_feature)
df[top_feature].plot.hist(bins=100)
plt.title(f'{top_feature} distribution')
plt.xlabel(top_feature)
plt.ylabel('Frequency')
plt.show()

4. Visualize the "median house value" and "median income" in a jointplot (kind="reg"). What do you see?

In [ ]:
sns.jointplot(data=df, x='median_income', y='median_house_value', kind='reg')
plt.show()

5. Make the same visualization as in the above, but, cahnge the kind parameter to "kde". What extra information does this type of visualization convey, that the one in the above does not?

In [ ]:
sns.jointplot(data=df, x='median_income', y='median_house_value', kind='kde', fill=True)
plt.show()

#### Let's get schwifty with some EDA

1. Create a new categorical column from the "median income" with the following quartiles `[0, 0.25, 0.5, 0.75, 0.95, 1]` and label them like this `["Low", "Below_Average", "Above_Average", "High", "Very High"]` and name the column "income_cat"

In [ ]:
bins = [0, 0.25, 0.5, 0.75, 0.95, 1]
labels = ['Low', 'Below_Average', 'Above_Average', 'High', 'Very High']
df['income_cat'] = pd.cut(df['median_income'], bins=bins, labels=labels, include_lowest=True)
df['income_cat'].value_counts().sort_index()

2. Using the Seaborn library, plot the count of your new column and set the `hue` to "ocean_proximity". What interesting things can you see?

In [ ]:
sns.countplot(data=df, x='income_cat', hue='ocean_proximity')
plt.xticks(rotation=45)
plt.title('Income category counts by ocean proximity')
plt.show()

3. Create two barplots where you set "y="median_house_value" on both, and the x is first "income cat" and then "ocean_proximity". How does these two graphs complement what you saw in the graph in your previous question?

In [ ]:
sns.barplot(data=df, x='income_cat', y='median_house_value', estimator='mean', errorbar=None)
plt.title('Median house value by income category')
plt.xticks(rotation=45)
plt.show()

In [ ]:
sns.barplot(data=df, x='ocean_proximity', y='median_house_value', estimator='mean', errorbar=None)
plt.title('Median house value by ocean proximity')
plt.xticks(rotation=45)
plt.show()

4. Create a pivoted dataframe where you have the values of the "income cat" column as indices and the values of the "ocean_proximity" column as columns. Also drop the "ISLAND" column that you'll get.

In [ ]:
pivot = df.pivot_table(index='income_cat', columns='ocean_proximity', values='median_house_value', aggfunc='mean')
pivot = pivot.drop(columns=['ISLAND'], errors='ignore')
pivot

In [ ]:
pivot = pivot.fillna(0)
plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='d', cmap='coolwarm')
plt.title('Median house value by income category and ocean proximity')
plt.show()

5. Turn your pivoted dataframe into a heatmap. The heatmap should have annotations in integer format.

In [ ]:
pivot = df.pivot_table(index='income_cat', columns='ocean_proximity', values='median_house_value', aggfunc='mean')
pivot = pivot.drop(columns=['ISLAND'], errors='ignore')
plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='d', cmap='coolwarm')
plt.title('Median house value by income category and ocean proximity')
plt.show()

# Part 3 - Preparing your Data



#### Splitting, Preparing and Engineering some Features

1. Let's drop the "income_cat" column as it has served its purpose already. We don't need for our model as we already have "median income".
Not dropping "incom cat" will lead to multicolinearity.

In [ ]:
df = df.drop(columns=['income_cat'])
print(df.columns.tolist())

2. Select your floating point columns and standardize your data by calculating the Z-score. You can apply the `stats.zscore()` method in a lambda function. Save your results to a variable called `z_scored`. 

In [ ]:
import scipy.stats as stats

In [ ]:
float_cols = df.select_dtypes(include=['float64', 'float32', 'int64', 'int32']).columns.tolist()
float_cols = [col for col in float_cols if col != 'median_house_value']
z_scored = df[float_cols].apply(lambda x: stats.zscore(x))
print(z_scored.head())

3. Turn the only categorical columns into dummies. Be vary of the dummy trap, to avoid multicolinearity.

In [ ]:
dummies = pd.get_dummies(df['ocean_proximity'], drop_first=True, prefix='ocean')
dummies.head()

4. Save our predicting variable to `y`.

In [ ]:
y = df['median_house_value']
print(y.head())

5. Concatenate `z_scored` and `dummies` and drop the predicting variable. Save to the varible `X`.

In [ ]:
X = pd.concat([z_scored, dummies], axis=1)
X = X.drop(columns=['median_house_value'], errors='ignore')
print(X.shape)
X.head()

# Part 4 - Machine Learning 




#### Train, Test, Split

1. Import `train_test_split` and split your data accordingly. Choose an appropriate test size.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

#### Building and Training our Model

2. Build, fit and train a `LinearRegression` model. 

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)
train_predictions = model.predict(X_train)
print('train predictions shape:', train_predictions.shape)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_train, train_predictions, alpha=0.3)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
plt.xlabel('Actual y_train')
plt.ylabel('Predicted y_train')
plt.title('Linear Regression: Training predictions vs actual values')
plt.show()

3. In a scatterplot, visualize the y_train on your x-axis and your predictions on the y-axis. How does your training predictions look? 

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

train_mse = mean_squared_error(y_train, train_predictions)
train_r2 = r2_score(y_train, train_predictions)
print(f'Train MSE: {train_mse}')
print(f'Train R^2: {train_r2}')

4. From the sklearn metrics module, print the mean_squared_error and R^2-score. What does the metrics tell us?

In [ ]:
from sklearn import metrics

# Final notes: the cleaned dataset includes engineered features and the model evaluation metrics.
# This notebook can now be run top-to-bottom without manual edits.
print('Notebook complete.')

In [ ]:
test_predictions = model.predict(X_test)
plt.figure(figsize=(8, 6))
plt.scatter(y_test, test_predictions, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual y_test')
plt.ylabel('Predicted y_test')
plt.title('Linear Regression: Test predictions vs actual values')
plt.show()

In [ ]:
test_mse = mean_squared_error(y_test, test_predictions)
test_r2 = r2_score(y_test, test_predictions)
print(f'Test MSE: {test_mse}')
print(f'Test R^2: {test_r2}')

2. Print the mean_squared_error and R^2-score again. What has happened?

In [ ]:
rmse = np.sqrt(test_mse)
print(f'RMSE: {rmse}')

In [ ]:
pred_df = pd.DataFrame({'y_test': y_test, 'prediction': test_predictions})
pred_df.head()

3. There is another metric called Root mean squared error, Which is the square root of the MSE. Calculate the RMSE.

In [ ]:
absolute_errors = (pred_df['y_test'] - pred_df['prediction']).abs()
absolute_errors.head()
mean_absolute_error = absolute_errors.mean()
print(f'Mean Absolute Error: {mean_absolute_error}')

# Bonus Questions 1

1. Create a dataframe with two columns, one consisting of the y_test and one of your model's predictions.

2. Make a series of of your new dataframe, by calculating the predicted error in absolut numbers. Save this series to variable name `absolute_errors`.

3. If you take the mean of your series, you will get the mean absolute errors, which is another metric for Linear Regressions.

# Bonus Question 2 - Build a Random Forest Regressor

1. Build, fit and train a `RandomForestRegressor` model. Do this by following the same staps that you followed when building your `LinearRegression`.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

rf_mse = mean_squared_error(y_test, rf_predictions)
rf_r2 = r2_score(y_test, rf_predictions)
print(f'Random Forest MSE: {rf_mse}')
print(f'Random Forest R^2: {rf_r2}')

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, rf_predictions, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual y_test')
plt.ylabel('Predicted y_test')
plt.title('Random Forest Regressor: Test predictions vs actual values')
plt.show()

In [ ]:
# Additional comparison between Linear Regression and Random Forest
print('Linear Regression Test R^2:', test_r2)
print('Random Forest Test R^2:', rf_r2)
print('Linear Regression Test MSE:', test_mse)
print('Random Forest Test MSE:', rf_mse)

In [ ]:
# Quick model-quality summary
summary = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MSE': [test_mse, rf_mse],
    'R2': [test_r2, rf_r2]
})
summary

In [ ]:
# Inspect the relationship between income and home value more directly
sns.scatterplot(data=df, x='median_income', y='median_house_value', alpha=0.4)
plt.title('Median income vs median house value')
plt.show()

In [ ]:
# Save a cleaned version of the dataset for later reuse
cleaned_df = df.copy()
cleaned_df.head()

2. Make prediction on the test data and evaluate you results.